# LEG-SEIR: train from audited public inputs

This standalone notebook trains all 15 configurations from scratch. The scientific model, folds, loss, training and selection cells are unchanged from the recorded experiment. The input loader is new: it uses pinned audited tables rather than parsing original PDFs. Consequently this notebook has a distinct code/input identity. Full retraining was not performed for the release; archived experiments have a separate independently verified identity.

On CPU, install `requirements-cpu.txt`. On Kaggle GPU, keep CUDA-enabled PyTorch and use `requirements-training.txt` for the recorded non-PyTorch dependencies; do not install CPU PyTorch. Enable GPU and Internet. Save this notebook before running; set `STUDY_NOTEBOOK` to its path if needed. Default: full 300-epoch maximum, two development folds, 3 development seeds and 5 final seeds. `LEG_SEIR_SMOKE=1` runs only two epochs and one seed to check execution; those outputs are **not paper results**. Full training takes substantially longer on CPU. No saved weights or forecasts are downloaded or read by training.

The 2026 evaluation is retrospective. Further model choices informed by these outcomes cannot be described as prospective confirmation. Do not change the unresolved-report policy silently.


In [ ]:
from pathlib import Path
import os,sys,datetime as dt,hashlib,json,time,random,math,copy,zipfile,platform,subprocess
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
import scipy,sklearn,joblib
from huggingface_hub import hf_hub_download
LOCK = {'repo_id': 'rathishTharusha/leg-seir', 'revision': '9c554d043563cb48d26aa047fefe58bea91597e6', 'files': [{'path': 'data/cases_25.csv', 'bytes': 86414, 'sha256': '9f2fab279a47ff56508fae6880b346735d5f1d15a3d860f970913a55394ac847'}, {'path': 'data/cases_25_printed_repeat_sensitivity.csv', 'bytes': 86620, 'sha256': '84ff1108955fef2a38f8a2dde79de132c45a79058726b5c48b5c241089814b55'}, {'path': 'data/cases_26.csv', 'bytes': 89125, 'sha256': 'b8d7cbb8b56668e5af1ce4311bf4d52765a4f4a4cfc77aa333f0a29b4eb340d7'}, {'path': 'data/population_audit.json', 'bytes': 900, 'sha256': '817623e1b1cd46beff89aea4bf04a5018151d5d59bd649d3bdfee7b09879bf1a'}, {'path': 'provenance/graph_audit.json', 'bytes': 4372, 'sha256': 'c889ddbd93e2f8c324302d7e6e676d5de4707a4a956c078327031b709516985d'}, {'path': 'provenance/source_manifest.json', 'bytes': 271337, 'sha256': '3976e939dffa0b0a721195babc835639f4084c6a3c564021b0ae7959b752541c'}, {'path': 'provenance/target_ledger.json', 'bytes': 839339, 'sha256': '441f375b3a17732ed5d9b369fc3f09e1f9549d7c650696dfca9f278ff7357762'}]}

INPUT=Path(os.environ.get('LEG_SEIR_DATA_DIR','leg_seir_inputs')).resolve()
INPUT.mkdir(parents=True,exist_ok=True)
def file_sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
for item in LOCK['files']:
 p=INPUT/item['path']
 if not p.exists():
  cached=Path(hf_hub_download(LOCK['repo_id'],item['path'],repo_type='dataset',revision=LOCK['revision']))
  p.parent.mkdir(parents=True,exist_ok=True);p.write_bytes(cached.read_bytes())
 assert file_sha(p)==item['sha256'], 'Input hash mismatch: '+str(p)
BASE=Path(os.environ.get('LEG_SEIR_RUN_DIR','runs/'+dt.datetime.now(dt.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))).resolve()
OUT=BASE/'evaluation_outputs'
assert not OUT.exists(), 'Use a fresh run directory; never overwrite evidence'
OUT.mkdir(parents=True)
for source,target in [('data/cases_25.csv','cases_25.csv'),('data/cases_26.csv','cases_26.csv'),('data/cases_25_printed_repeat_sensitivity.csv','cases_25_printed_repeat_sensitivity.csv'),('data/population_audit.json','population_audit.json'),('provenance/target_ledger.json','target_ledger.json'),('provenance/source_manifest.json','source_manifest.json'),('provenance/graph_audit.json','graph_audit.json')]:
 (OUT/target).write_bytes((INPUT/source).read_bytes())
frame=pd.read_csv(INPUT/'data/cases_25.csv')
NAMES=list(frame.columns[:25]);CASES=frame[NAMES].to_numpy(dtype=np.float64)
records=json.loads((INPUT/'provenance/target_ledger.json').read_text())
population=json.loads((INPUT/'data/population_audit.json').read_text())['census2012']
POP=np.array([population[n] for n in NAMES],dtype=np.float32)
N,WINDOW,HORIZON,WIDTH=25,3,3,64
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_num_threads(1);torch.set_num_interop_threads(1)
PROFILE='exploration';STAGE='state_observer';SMOKE=False
EVALUATION_SMOKE=os.environ.get('LEG_SEIR_SMOKE','0')=='1'
nbpath=Path(os.environ.get('STUDY_NOTEBOOK','/kaggle/working/__notebook__.ipynb' if Path('/kaggle/working/__notebook__.ipynb').exists() else 'notebooks/LEG_SEIR_train_from_audited_data.ipynb'))
if not nbpath.exists(): nbpath=Path('LEG_SEIR_train_from_audited_data.ipynb')
assert nbpath.is_file(), 'Set STUDY_NOTEBOOK to the saved notebook path'
nb=json.loads(nbpath.read_text(encoding='utf8'))
codes=[''.join(c['source']) for c in nb['cells'] if c['cell_type']=='code']
CODE_HASH=hashlib.sha256(json.dumps(codes,sort_keys=True).encode()).hexdigest()
DATA_HASH=hashlib.sha256(json.dumps(LOCK,sort_keys=True).encode()).hexdigest()
(OUT/'submitted_notebook.ipynb').write_bytes(nbpath.read_bytes())
(OUT/'audited_input_identity.json').write_text(json.dumps(LOCK,indent=2))
(OUT/'pip_freeze.txt').write_bytes(subprocess.check_output([sys.executable,'-m','pip','freeze']))
print('Device:',DEVICE,'Smoke:',EVALUATION_SMOKE,'Fresh output:',OUT,'New code hash:',CODE_HASH)


## Inputs, normalization and metrics

In [ ]:
def pack(fold,split,cases=CASES):
    ids=fold['idx'][split]
    rawx=np.stack([cases[i-WINDOW:i].T for i in ids]).astype(np.float32)
    rawy=np.stack([cases[i:i+HORIZON].T for i in ids]).astype(np.float64)
    x=(np.log1p(rawx)-fold['mean'])/fold['std']
    y=(np.log1p(rawy)-fold['mean'])/fold['std']
    return {'idx':ids,'x':torch.tensor(x,device=DEVICE),'y':torch.tensor(y,dtype=torch.float32,device=DEVICE),
            'p':torch.tensor(np.repeat(x[:,:,-1:],HORIZON,2),device=DEVICE),
            'rawx':torch.tensor(rawx,device=DEVICE),'truth':rawy}
def counts(z,fold):
    y=np.expm1(np.asarray(z,dtype=np.float64)*fold['std']+fold['mean'])
    assert np.isfinite(y).all(), 'Nonfinite inverse forecast'
    return np.maximum(y,0)
def metrics(pred,true):
    err=pred-true
    den=np.abs(pred)+np.abs(true)
    smape=np.divide(200*np.abs(err),den,out=np.zeros_like(err),where=den>0)
    return {'RMSE':float(np.sqrt(np.mean(err**2))),'MAE':float(np.abs(err).mean()),'sMAPE':float(smape.mean())}
def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark=False
def predict_z(model,data):
    model.eval()
    with torch.no_grad():
        return model(data['x'],data['p'],data['rawx']).cpu().numpy()


## Conservative mechanistic decoder

In [ ]:
def conservative_step(state,force,gamma=1/7,omega=.1):
    s,e,i,r=state.unbind(-1)
    u=s*(-torch.expm1(-force))
    v=e*(-math.expm1(-omega))
    w=i*(-math.expm1(-gamma))
    return torch.stack([s-u,e+u-v,i+v-w,r+w],-1),v

def mechanistic_forecast(rawx,raw,feedback=False,gamma=1/7):
    scale=torch.as_tensor(POP,device=rawx.device,dtype=rawx.dtype)[None]/11
    i=(rawx[:,:,-1]/scale).clamp(1e-6,.5)
    e=(rawx[:,:,-2]/scale).clamp(1e-6,.5)
    s=torch.full_like(i,.318)
    r=(1-s-e-i).clamp_min(0)
    state=torch.stack([s,e,i,r],-1)
    state=state/state.sum(-1,keepdim=True)
    prediction=[]
    for h in range(HORIZON):
        # Separate parameterizations: old neural force versus neural beta.
        rate=torch.exp((raw[:,:,h]-(1.1 if feedback else 9.36)).clamp(*((-9,1.1) if feedback else (-18,-1.95))))
        total=torch.zeros_like(i)
        for _ in range(7):
            force=rate*state[:,:,2] if feedback else rate
            state,onset=conservative_step(state,force,gamma=gamma)
            total=total+onset
        prediction.append(total*scale)
    return torch.stack(prediction,-1)



## Shared neural and graph components

In [ ]:
"""Local-preserving graph prototypes; embedded in each standalone notebook."""


class Candidate(nn.Module):
    def __init__(self, config, fold):
        super().__init__()
        self.config, self.fold = dict(config), fold
        self.identity = config.get("identity", True)
        self.differences = config.get("differences", True)
        self.graph = config.get("graph", True)
        self.mode = config.get("mode", "lgi")
        self.decoder = config.get("decoder", "seir")
        self.use_router = self.mode == "router" or (
            self.mode == "adapters" and config.get("routing", True)
        )
        self.embedding = nn.Parameter(torch.randn(N, 4) * 0.05) if self.identity else None
        self.first = nn.Linear(
            WINDOW + (WINDOW - 1 if self.differences else 0) + (4 if self.identity else 0), WIDTH
        )
        self.next = nn.Linear(WIDTH, WIDTH)
        self.local_head = nn.Linear(WIDTH, HORIZON)
        if self.mode == "adapters":
            self.adapter_rank = config.get("adapter_rank", 8)
            self.district_conditioned = config.get("district_conditioned", True)
            self.adapter_down = nn.Linear(WIDTH, self.adapter_rank, bias=False)
            self.adapter_up = nn.Linear(self.adapter_rank, WIDTH, bias=False)
            nn.init.zeros_(self.adapter_up.weight)
            self.adapter_gate = nn.Parameter(torch.tensor(-2.0))
            if self.district_conditioned:
                self.adapter_embedding = nn.Parameter(torch.randn(N, 4) * 0.05)
                self.adapter_condition = nn.Linear(4, self.adapter_rank, bias=False)
            else:
                self.adapter_coefficients = nn.Parameter(torch.full((self.adapter_rank,), 0.05))
        if self.mode == "growth":
            # Preserve level information locally while graph messages see only changes.
            self.level_projection = nn.Linear(1, WIDTH, bias=False)
        if self.graph:
            self.left = nn.Parameter(torch.randn(N, 8) * 0.1)
            self.right = nn.Parameter(torch.randn(N, 8) * 0.1)
            self.message = nn.Linear(WIDTH, WIDTH, bias=False)
            self.graph_gate = nn.Parameter(torch.full((1,), -2.0))
            if self.mode == "channels":
                self.channel_gate = nn.Linear(WIDTH, 3)
        if self.decoder != "none":
            self.rate_head = nn.Linear(WIDTH, HORIZON)
            if not self.use_router:
                self.mix = nn.Parameter(torch.full((HORIZON,), -2.0))
                self.persistence_gate = nn.Parameter(torch.full((HORIZON,), 2.0))
        if self.use_router and self.decoder != "none":
            self.router_style = config.get(
                "router_style", "horizon" if self.mode == "adapters" else "conditional"
            )
            if self.router_style == "conditional":
                self.router = nn.Linear(WIDTH, HORIZON * 4)
                nn.init.zeros_(self.router.weight)
                nn.init.zeros_(self.router.bias)
            elif self.router_style == "horizon":
                self.router_logits = nn.Parameter(torch.zeros(HORIZON, 4))
            elif self.router_style == "fixed":
                self.register_buffer("fixed_router_weights", torch.full((HORIZON, 4), 0.25))
            else:
                raise ValueError(self.router_style)

    def adjacency(self):
        scores = self.left @ self.right.T
        mask = torch.eye(N, device=scores.device, dtype=torch.bool)
        scores = scores.masked_fill(mask, -torch.inf)
        top = scores.topk(5, dim=-1).indices
        keep = torch.zeros_like(scores, dtype=torch.bool).scatter(1, top, True)
        return torch.softmax(scores.masked_fill(~keep, -torch.inf), -1)

    def base_hidden(self, x):
        v = x - x[:, :, -1:] if self.mode == "growth" else x
        features = [v]
        if self.differences:
            features.append(x[:, :, 1:] - x[:, :, :-1])
        if self.identity:
            features.append(self.embedding[None].expand(x.shape[0], -1, -1))
        return F.relu(self.next(F.relu(self.first(torch.cat(features, -1)))))

    def adapter_coeffs(self):
        if self.district_conditioned:
            return torch.tanh(self.adapter_condition(self.adapter_embedding))
        return torch.tanh(self.adapter_coefficients)[None].expand(N, -1)

    def adapter_delta(self, h):
        return (
            0.1
            * torch.sigmoid(self.adapter_gate)
            * self.adapter_up(self.adapter_down(h) * self.adapter_coeffs()[None])
        )

    def local_hidden(self, x, adapter_scale=1.0):
        h = self.base_hidden(x)
        return h + adapter_scale * self.adapter_delta(h) if self.mode == "adapters" else h

    def routing_weights(self, h):
        if self.router_style == "conditional":
            return torch.softmax(self.router(h).reshape(*h.shape[:2], HORIZON, 4), -1)
        weights = (
            torch.softmax(self.router_logits, -1)
            if self.router_style == "horizon"
            else self.fixed_router_weights
        )
        return weights[None, None].expand(h.shape[0], h.shape[1], -1, -1)

    def forward(self, x, p, rawx, graph_scale=1.0, return_details=False, adapter_scale=1.0):
        h = self.local_hidden(x, adapter_scale=adapter_scale)
        level = self.level_projection(x[:, :, -1:]) if self.mode == "growth" else 0.0
        local = self.local_head(h + level) + (p if self.mode == "growth" else 0.0)
        hg = h
        if self.graph:
            avg = torch.einsum("ij,bjw->biw", self.adjacency(), h)
            delta = avg - h
            if self.mode == "channels":
                weights = torch.softmax(self.channel_gate(h), -1)
                delta = weights[:, :, 1:2] * avg + weights[:, :, 2:3] * (h - avg)
            hg = h + graph_scale * torch.sigmoid(self.graph_gate).mean() * self.message(delta)
        if self.decoder == "none":
            return self.local_head(hg + level) + (p if self.mode == "growth" else 0.0)
        raw = self.rate_head(hg + level)
        if self.decoder == "seir":
            y = mechanistic_forecast(rawx, raw, feedback=True)
            component = (torch.log1p(y) - self.fold["mean"]) / self.fold["std"]
        else:
            component = raw + (p if self.mode == "growth" else 0.0)
        if self.use_router:
            weights = self.routing_weights(h)
            graph_forecast = self.local_head(hg)
            experts = torch.stack([p, local, graph_forecast, component], -1)
            result = (weights * experts).sum(-1)
            return (result, weights, experts) if return_details else result
        base = p + torch.sigmoid(self.persistence_gate) * (local - p)
        return base + torch.sigmoid(self.mix) * (component - base)


## Lagged-exposure graph-SEIR

In [ ]:
"""Lagged exposure: sparse sender graph, lag mixture and daily imported pressure."""

BaseCandidate = Candidate


class Candidate(BaseCandidate):
    def __init__(self, config, fold):
        self.is_lagged = config.get("mode") == "lagged"
        cfg = dict(config)
        if self.is_lagged:
            cfg["mode"] = "router" if config.get("routing", True) else "lgi"
        super().__init__(cfg, fold)
        self.config = dict(config)
        if self.is_lagged and self.graph:
            # Per-recipient convex mixture over the same three observed report lags.
            self.lag_logits = nn.Parameter(torch.zeros(N, WINDOW))
            if self.decoder == "seir" and config.get("exposure", True):
                self.import_gate = nn.Parameter(torch.tensor(-2.0))

    def lag_weights(self):
        return torch.softmax(self.lag_logits, -1)

    def imported(self, rawx):
        scale = torch.as_tensor(POP, device=rawx.device, dtype=rawx.dtype)[None] / 11
        fractions = (rawx / scale[:, :, None]).clamp(1e-6, 0.5)
        spatial = torch.einsum("ij,bjl->bil", self.adjacency(), fractions)
        return (spatial * self.lag_weights()[None]).sum(-1)

    def exposure_forecast(self, rawx, rates, graph_scale=1.0, return_state=False):
        scale = torch.as_tensor(POP, device=rawx.device, dtype=rawx.dtype)[None] / 11
        infectious = (rawx[:, :, -1] / scale).clamp(1e-6, 0.5)
        exposed = (rawx[:, :, -2] / scale).clamp(1e-6, 0.5)
        susceptible = torch.full_like(infectious, 0.318)
        recovered = (1 - susceptible - exposed - infectious).clamp_min(0)
        state = torch.stack([susceptible, exposed, infectious, recovered], -1)
        state = state / state.sum(-1, keepdim=True)
        mass = state.sum(-1)
        outputs, states = [], []
        gate = (
            graph_scale * torch.sigmoid(self.import_gate) if hasattr(self, "import_gate") else 0.0
        )
        # Historical pressure remains a decaying observation anchor. Daily evolving
        # neighbours progressively replace it; no future reports are consumed.
        anchor = self.imported(rawx) if self.graph else infectious
        for h in range(HORIZON):
            beta = torch.exp((rates[:, :, h] - 1.1).clamp(-9, 1.1))
            total = torch.zeros_like(infectious)
            for day in range(7):
                elapsed = h * 7 + day
                neighbours = (
                    torch.einsum("ij,bj->bi", self.adjacency(), state[:, :, 2])
                    if self.graph
                    else state[:, :, 2]
                )
                decay = math.exp(-elapsed / 7)
                pressure = decay * anchor + (1 - decay) * neighbours
                force = beta * ((1 - gate) * state[:, :, 2] + gate * pressure)
                state, onset = conservative_step(state, force)
                total = total + onset
                states.append(state)
            outputs.append(total * scale)
        result = torch.stack(outputs, -1)
        return (result, torch.stack(states, -2), mass) if return_state else result

    def forward(self, x, p, rawx, graph_scale=1.0, return_details=False, **kwargs):
        if not self.is_lagged:
            return super().forward(x, p, rawx, graph_scale, return_details, **kwargs)
        h = self.base_hidden(x)
        hg = h
        if self.graph:
            lagged = torch.einsum("ij,bjl->bil", self.adjacency(), x)
            # Learned lag selection also informs the neural branch. Each lag retains
            # its own temporal coordinate; positive weights sum to one per recipient.
            selected = lagged * self.lag_weights()[None] * WINDOW
            neighbour_h = self.base_hidden(selected)
            hg = h + graph_scale * torch.sigmoid(self.graph_gate) * self.message(neighbour_h - h)
        local = self.local_head(h)
        if self.decoder == "none":
            return self.local_head(hg)
        raw = self.rate_head(hg)
        component = (
            (torch.log1p(self.exposure_forecast(rawx, raw, graph_scale)) - self.fold["mean"])
            / self.fold["std"]
            if self.decoder == "seir"
            else raw
        )
        if self.use_router:
            weights = self.routing_weights(h)
            experts = torch.stack([p, local, self.local_head(hg), component], -1)
            result = (weights * experts).sum(-1)
            return (result, weights, experts) if return_details else result
        base = p + torch.sigmoid(self.persistence_gate) * (local - p)
        return base + torch.sigmoid(self.mix) * (component - base)


## Observer comparator

In [ ]:
"""State observer: bounded local simplex correction before conservative SEIR."""

BaseCandidate = Candidate


class Candidate(BaseCandidate):
    def __init__(self, config, fold):
        self.is_observer = config.get("mode") == "observer"
        cfg = dict(config)
        if self.is_observer:
            cfg["mode"] = "router" if config.get("routing", True) else "lgi"
        super().__init__(cfg, fold)
        self.config = dict(config)
        if self.is_observer:
            self.observer_head = nn.Linear(WIDTH, 4)
            nn.init.zeros_(self.observer_head.weight)
            nn.init.zeros_(self.observer_head.bias)
            if config.get("state_features", True):
                self.state_projection = nn.Linear(4, WIDTH, bias=False)

    def prior_state(self, rawx):
        scale = torch.as_tensor(POP, device=rawx.device, dtype=rawx.dtype)[None] / 11
        i = (rawx[:, :, -1] / scale).clamp(1e-6, 0.5)
        e = (rawx[:, :, -2] / scale).clamp(1e-6, 0.5)
        s = torch.full_like(i, 0.318)
        r = (1 - s - e - i).clamp_min(0)
        state = torch.stack([s, e, i, r], -1)
        return state / state.sum(-1, keepdim=True)

    def corrected_state(self, x, rawx, observer_scale=1.0):
        prior = self.prior_state(rawx)
        if observer_scale == 0:
            return prior, prior, torch.zeros_like(prior)
        correction = observer_scale * 0.5 * torch.tanh(self.observer_head(self.base_hidden(x)))
        state = prior * torch.exp(correction)
        state = state / state.sum(-1, keepdim=True)
        return state, prior, correction

    def observer_forecast(self, state, rates, return_states=False):
        scale = torch.as_tensor(POP, device=state.device, dtype=state.dtype)[None] / 11
        prediction, states = [], []
        for h in range(HORIZON):
            beta = torch.exp((rates[:, :, h] - 1.1).clamp(-9, 1.1))
            total = torch.zeros_like(state[:, :, 0])
            for _ in range(7):
                state, onset = conservative_step(state, beta * state[:, :, 2])
                total = total + onset
                states.append(state)
            prediction.append(total * scale)
        y = torch.stack(prediction, -1)
        return (y, torch.stack(states, -2)) if return_states else y

    def forward(
        self, x, p, rawx, graph_scale=1.0, return_details=False, observer_scale=1.0, **kwargs
    ):
        if not self.is_observer:
            return super().forward(x, p, rawx, graph_scale, return_details, **kwargs)
        state, prior, _ = self.corrected_state(x, rawx, observer_scale)
        h = self.base_hidden(x)
        if hasattr(self, "state_projection"):
            h = h + self.state_projection(state - prior)
        hg = h
        if self.graph:
            avg = torch.einsum("ij,bjw->biw", self.adjacency(), h)
            hg = h + graph_scale * torch.sigmoid(self.graph_gate) * self.message(avg - h)
        local = self.local_head(h)
        if self.decoder == "none":
            return self.local_head(hg)
        rates = self.rate_head(hg)
        component = (
            (torch.log1p(self.observer_forecast(state, rates)) - self.fold["mean"])
            / self.fold["std"]
            if self.decoder == "seir"
            else rates
        )
        if self.use_router:
            weights = self.routing_weights(h)
            experts = torch.stack([p, local, self.local_head(hg), component], -1)
            y = (weights * experts).sum(-1)
            return (y, weights, experts) if return_details else y
        base = p + torch.sigmoid(self.persistence_gate) * (local - p)
        return base + torch.sigmoid(self.mix) * (component - base)


## Frozen configurations and matched removals

In [ ]:
"""Frozen shortlist and attribution controls; embedded in evaluation notebook."""

common = {"identity": False, "differences": True, "router_style": "horizon"}
CONFIGS = []
for family, mode, extra in [
    ("horizon", "router", {}),
    ("lagged", "lagged", {}),
    ("observer", "observer", {"state_features": False}),
]:
    for removal in ["full", "graph_off", "seir_off"]:
        # Without SEIR, compartment-only assimilation has no remaining operation.
        # Its replacement is the ordinary neural horizon mixture, not dead observer parameters.
        actual_mode = "router" if family == "observer" and removal == "seir_off" else mode
        CONFIGS.append(
            {
                **common,
                **extra,
                "name": family + "_" + removal,
                "mode": actual_mode,
                "graph": removal != "graph_off",
                "decoder": "direct" if removal == "seir_off" else "seir",
                "parent": family + "_full",
                "role": "candidate" if removal == "full" else "removal",
            }
        )
CONFIGS += [
    {
        **common,
        "name": "local_mlp",
        "mode": "lgi",
        "graph": False,
        "decoder": "none",
        "role": "baseline",
    },
    {
        **common,
        "name": "identity_mlp",
        "mode": "lgi",
        "identity": True,
        "graph": False,
        "decoder": "none",
        "role": "baseline",
    },
    {
        **common,
        "name": "local_seir",
        "mode": "lgi",
        "graph": False,
        "decoder": "seir",
        "role": "baseline",
    },
    {
        **common,
        "name": "plain_graph",
        "mode": "lgi",
        "graph": True,
        "decoder": "none",
        "role": "baseline",
    },
    {
        **common,
        "name": "uniform_mixture",
        "mode": "router",
        "graph": True,
        "decoder": "seir",
        "router_style": "fixed",
        "role": "baseline",
    },
    {
        **common,
        "name": "shared_adapter",
        "mode": "adapters",
        "graph": True,
        "decoder": "seir",
        "district_conditioned": False,
        "adapter_rank": 8,
        "adapter_no_decay": True,
        "role": "baseline",
    },
]
assert len(CONFIGS) == 15 and len({c["name"] for c in CONFIGS}) == 15


## Development selection, final fits and complete exports

In [ ]:
"""Full development selection precedes every 2026 retrospective forecast."""

EPOCHS, PATIENCE, BATCH = (2, 2, 128) if EVALUATION_SMOKE else (300, 40, 128)
DEV_SEEDS = [0] if EVALUATION_SMOKE else [0, 1, 2]
FINAL_SEEDS = [0] if EVALUATION_SMOKE else [0, 1, 2, 3, 4]
LRS = [0.003] if EVALUATION_SMOKE else [0.001, 0.003]
if Path("/kaggle/input").exists():
    assert DEVICE.type == "cuda"
YEARS = np.array([r["clinical_year"] for r in records])
valid = [i for i in range(3, len(CASES) - 2) if np.isfinite(CASES[i - 3 : i + 3]).all()]


def make_evaluation_fold(year, test=False):
    idx = {
        "train": np.array([i for i in valid if (YEARS[i : i + 3] < year).all()]),
        "val": np.array([i for i in valid if (YEARS[i : i + 3] == year).all()]),
    }
    if test:
        assert (OUT / "learning_rate_freeze.json").exists(), "Freeze before creating test tensors"
        idx["test"] = np.array([i for i in valid if (YEARS[i : i + 3] == 2026).all()])
    assert all(len(v) for v in idx.values())
    assert idx["train"].max() + 2 < idx["val"].min()
    if test:
        assert idx["val"].max() + 2 < idx["test"].min()
    touched = sorted({j for i in idx["train"] for j in range(i - 3, i + 3)})
    log = np.log1p(CASES[touched])
    return {"idx": idx, "mean": float(log.mean()), "std": float(log.std() + 1e-8)}


PLAN = {
    "status": "frozen_before_fit",
    "code_hash": CODE_HASH,
    "data_hash": DATA_HASH,
    "source_policy": "frozen v3 unchanged",
    "configs": CONFIGS,
    "development_years": [2023, 2025],
    "learning_rates": LRS,
    "development_seeds": DEV_SEEDS,
    "final_seeds": FINAL_SEEDS,
    "epochs": EPOCHS,
    "patience": PATIENCE,
    "batch": BATCH,
    "width": WIDTH,
    "selection": "maximum mean fold skill vs persistence across folds and seeds; exact tie lower LR",
    "primary_removals": "same selected LR as parent; separately selected removal LR sensitivity if different",
    "test": "2026 retrospective rolling forecasts, no parameter refit/tuning from 2026 targets",
    "source_cutoff": "attached version1 archive through volume53 report33, not complete calendar2026",
    "smoke": EVALUATION_SMOKE,
    "expected_development_fits": len(CONFIGS) * len(LRS) * len(DEV_SEEDS) * 2,
    "expected_primary_final_fits": len(CONFIGS) * len(FINAL_SEEDS),
    "max_extra_removal_sensitivity_fits": 6 * len(FINAL_SEEDS),
}
(OUT / "evaluation_plan.json").write_text(json.dumps(PLAN, indent=2))
(OUT / "environment.json").write_text(
    json.dumps(
        {
            "python": sys.version,
            "torch": torch.__version__,
            "device": str(DEVICE),
            "numpy": np.__version__,
            "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
        },
        indent=2,
    )
)
rows, trials = [], []


def ridge_baseline(fold, splits, token):
    training = pack(fold, "train")

    def features(data):
        x = data["x"].cpu().numpy()
        return np.concatenate([x, np.diff(x, axis=-1)], -1).reshape(-1, 5)

    fitted = Ridge(alpha=1).fit(features(training), training["y"].cpu().numpy().reshape(-1, 3))
    joblib.dump(fitted, OUT / (token + ".joblib"))
    for split in splits:
        data = pack(fold, split)
        z = fitted.predict(features(data)).reshape(len(data["idx"]), 25, 3)
        np.savez_compressed(
            OUT / (token + "_" + split + ".npz"),
            idx=data["idx"],
            truth=data["truth"],
            prediction=counts(z, fold),
        )


def baseline(fold, split, token):
    ids = fold["idx"][split]
    truth = np.stack([CASES[i : i + 3].T for i in ids])
    pred = np.repeat(CASES[ids - 1, :, None], 3, 2)
    np.savez_compressed(OUT / (token + ".npz"), idx=ids, truth=truth, prediction=pred)
    return metrics(pred, truth)


def fit(config, fold, lr, seed, token, phase, year, sensitivity=False):
    seed_all(seed)
    tr, va = pack(fold, "train"), pack(fold, "val")
    model = Candidate(config, fold).to(DEVICE)
    if config.get("adapter_no_decay"):
        groups = [
            {
                "params": [p for n, p in model.named_parameters() if n.startswith("adapter_")],
                "weight_decay": 0.0,
            },
            {
                "params": [p for n, p in model.named_parameters() if not n.startswith("adapter_")],
                "weight_decay": 1e-4,
            },
        ]
        optimizer = torch.optim.Adam(groups, lr=lr)
    else:
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    best, stale, best_state, best_epoch = float("inf"), 0, None, 0
    curve = []
    for epoch in range(EPOCHS):
        model.train()
        perm = torch.randperm(len(tr["idx"]), device=DEVICE)
        losses = []
        for start in range(0, len(perm), BATCH):
            ix = perm[start : start + BATCH]
            optimizer.zero_grad(set_to_none=True)
            z = model(*[tr[k][ix] for k in ["x", "p", "rawx"]])
            loss = F.mse_loss(z, tr["y"][ix])
            assert torch.isfinite(loss)
            loss.backward()
            assert all(
                torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None
            )
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            losses.append((float(loss.detach().cpu()), len(ix)))
        score = metrics(counts(predict_z(model, va), fold), va["truth"])["RMSE"]
        curve.append(
            {
                "epoch": epoch + 1,
                "train_loss": sum(v * n for v, n in losses) / sum(n for _, n in losses),
                "validation_RMSE": score,
            }
        )
        if score < best - 1e-8:
            best, best_state, stale, best_epoch = (
                score,
                copy.deepcopy(model.state_dict()),
                0,
                epoch + 1,
            )
        else:
            stale += 1
        if stale >= PATIENCE:
            break
    assert best_state is not None
    model.load_state_dict(best_state)
    checkpoint = {
        "state_dict": {k: v.cpu() for k, v in best_state.items()},
        "config": config,
        "fold": fold,
        "lr": lr,
        "seed": seed,
        "code_hash": CODE_HASH,
        "data_hash": DATA_HASH,
        "phase": phase,
        "year": year,
        "best_epoch": best_epoch,
    }
    torch.save(checkpoint, OUT / (token + ".pt"))
    pd.DataFrame(curve).to_csv(OUT / (token + "_curve.csv"), index=False)
    # CPU reload and parity before its validation score can enter selection.
    cpu = Candidate(config, fold)
    cpu.load_state_dict(checkpoint["state_dict"])
    cpu.eval()
    with torch.no_grad():
        cz = cpu(*[va[k].cpu() for k in ["x", "p", "rawx"]]).numpy()
    np.testing.assert_allclose(
        counts(cz, fold), counts(predict_z(model, va), fold), rtol=1e-4, atol=0.001
    )
    splits = ["val"] if phase == "development" else ["val", "test"]
    for split in splits:
        data = va if split == "val" else pack(fold, split)
        pred = counts(predict_z(model, data), fold)
        arrays = {"idx": data["idx"], "truth": data["truth"], "prediction": pred}
        with torch.no_grad():
            if model.use_router and config["decoder"] != "none":
                _, w, experts = model(data["x"], data["p"], data["rawx"], return_details=True)
                arrays.update(router_weights=w.cpu().numpy(), router_expert_z=experts.cpu().numpy())
            if config["graph"]:
                arrays["adjacency"] = model.adjacency().detach().cpu().numpy()
                arrays["graph_off_prediction"] = counts(
                    model(data["x"], data["p"], data["rawx"], graph_scale=0).cpu().numpy(), fold
                )
            if config["mode"] == "lagged" and config["graph"]:
                arrays["lag_weights"] = model.lag_weights().detach().cpu().numpy()
                if hasattr(model, "import_gate"):
                    arrays["import_gate"] = torch.sigmoid(model.import_gate).detach().cpu().numpy()
            if config["mode"] == "observer":
                state, prior, correction = model.corrected_state(data["x"], data["rawx"])
                arrays.update(
                    observer_state=state.cpu().numpy(),
                    observer_prior=prior.cpu().numpy(),
                    observer_correction=correction.cpu().numpy(),
                )
        np.savez_compressed(OUT / (token + "_" + split + ".npz"), **arrays)
        result = metrics(pred, data["truth"])
        row = {
            "token": token,
            "name": config["name"],
            "role": config["role"],
            "phase": phase,
            "year": year,
            "split": split,
            "lr": lr,
            "seed": seed,
            "sensitivity": sensitivity,
            **result,
            "best_epoch": best_epoch,
            "epochs_run": len(curve),
            "parameters": sum(p.numel() for p in model.parameters()),
        }
        for h in range(3):
            row["RMSE_h" + str(h + 1)] = metrics(pred[:, :, h], data["truth"][:, :, h])["RMSE"]
        if phase == "development":
            persistence = metrics(np.repeat(CASES[data["idx"] - 1, :, None], 3, 2), data["truth"])[
                "RMSE"
            ]
            row["skill"] = 1 - result["RMSE"] / persistence
        rows.append(row)
    trials.append(
        {
            "token": token,
            "phase": phase,
            "config": config["name"],
            "lr": lr,
            "seed": seed,
            "year": year,
            "sensitivity": sensitivity,
        }
    )
    pd.DataFrame(rows).to_csv(OUT / "scores.csv", index=False)
    (OUT / "trials.json").write_text(json.dumps(trials, indent=2))
    print("FIT_COMPLETE", token, "validation RMSE", best, flush=True)


for year in [2023, 2025]:
    fold = make_evaluation_fold(year)
    baseline(fold, "val", "persistence_dev_" + str(year))
    ridge_baseline(fold, ["val"], "ridge_dev_" + str(year))
    for config in CONFIGS:
        for lr in LRS:
            for seed in DEV_SEEDS:
                token = f"dev_{year}_{config['name']}_lr{lr:g}_seed{seed}"
                fit(config, fold, lr, seed, token, "development", year)
assert len(trials) == PLAN["expected_development_fits"]
development = pd.DataFrame(rows)
assert (development.year < 2026).all() and (development.split == "val").all()
selection_scores = development.groupby(["name", "lr"]).skill.mean().reset_index()
selected = {}
for name in [c["name"] for c in CONFIGS]:
    group = selection_scores[selection_scores.name == name].sort_values(
        ["skill", "lr"], ascending=[False, True]
    )
    selected[name] = float(group.iloc[0].lr)
primary = {c["name"]: selected[c.get("parent", c["name"])] for c in CONFIGS}
freeze = {
    "status": "frozen_from_pre2026_only",
    "selected": selected,
    "primary": primary,
    "development_scores": selection_scores.to_dict("records"),
    "code_hash": CODE_HASH,
    "data_hash": DATA_HASH,
    "created_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "development_fits": len(trials),
    "test_predictions_at_freeze": 0,
}
(OUT / "learning_rate_freeze.json").write_text(json.dumps(freeze, indent=2))
print("LEARNING_RATES_FROZEN_BEFORE_TEST", json.dumps(primary), flush=True)
final_fold = make_evaluation_fold(2025, test=True)
baseline(final_fold, "val", "persistence_final_val")
baseline(final_fold, "test", "persistence_final_test")
ridge_baseline(final_fold, ["val", "test"], "ridge_final")
for config in CONFIGS:
    name = config["name"]
    settings = [(primary[name], False)]
    if config["role"] == "removal" and selected[name] != primary[name]:
        settings.append((selected[name], True))
    for lr, sensitivity in settings:
        for seed in FINAL_SEEDS:
            token = f"final_{name}_lr{lr:g}_seed{seed}" + ("_sensitivity" if sensitivity else "")
            fit(config, final_fold, lr, seed, token, "final", 2026, sensitivity)
summary = (
    pd.DataFrame(rows)
    .query("phase=='final' and split=='test' and sensitivity==False")
    .groupby("name")
    .agg(mean_RMSE=("RMSE", "mean"), seed_SD=("RMSE", "std"))
)
summary.to_csv(OUT / "test_summary.csv")
primary_rows = pd.DataFrame(rows).query("phase=='final' and split=='test' and sensitivity==False")
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name in ["horizon_full", "lagged_full", "observer_full"]:
    selected_rows = primary_rows[primary_rows.name == name]
    errors = selected_rows[["RMSE_h1", "RMSE_h2", "RMSE_h3"]].mean().to_numpy()
    axes[0].plot([1, 2, 3], errors, marker="o", label=name)
    arrays = [np.load(OUT / (token + "_test.npz")) for token in selected_rows.token]
    ensemble = np.mean([arr["prediction"] for arr in arrays], axis=0)
    np.savez_compressed(
        OUT / (name + "_test_ensemble.npz"),
        idx=arrays[0]["idx"],
        truth=arrays[0]["truth"],
        prediction=ensemble,
    )
    axes[1].plot(arrays[0]["idx"], ensemble[:, :, 0].sum(1), label=name)
axes[0].set(xlabel="Report-observation horizon", ylabel="Count RMSE; mean over seeds")
axes[1].plot(
    arrays[0]["idx"], arrays[0]["truth"][:, :, 0].sum(1), color="black", label="Observed total"
)
axes[1].set(xlabel="Report slot of forecast origin", ylabel="25-district total, horizon 1")
for ax in axes:
    ax.legend()
fig.suptitle("Retrospective 2026 case study; frozen pre-2026 settings")
fig.tight_layout()
for extension in ["png", "svg", "pdf"]:
    fig.savefig(OUT / ("case_study_horizons_totals." + extension), dpi=180)
plt.close(fig)
fig, ax = plt.subplots(figsize=(10, 6))
summary.sort_values("mean_RMSE", ascending=False).mean_RMSE.plot.barh(ax=ax)
ax.axvline(
    metrics(
        np.repeat(CASES[final_fold["idx"]["test"] - 1, :, None], 3, 2),
        pack(final_fold, "test")["truth"],
    )["RMSE"],
    color="red",
    label="Persistence",
)
ax.set_xlabel("Retrospective 2026 count RMSE; five-seed mean")
ax.legend()
fig.tight_layout()
for extension in ["png", "svg", "pdf"]:
    fig.savefig(OUT / ("retrospective_test_ranking." + extension), dpi=180)
plt.close(fig)
completion = {
    "status": "complete",
    "smoke": EVALUATION_SMOKE,
    "code_hash": CODE_HASH,
    "data_hash": DATA_HASH,
    "development_fits": PLAN["expected_development_fits"],
    "final_fits": len(trials) - PLAN["expected_development_fits"],
    "fits": len(trials),
    "files": [
        {"path": p.name, "sha256": file_sha(p)}
        for p in sorted(OUT.iterdir())
        if p.is_file() and p.name != "completion.json"
    ],
}
(OUT / "completion.json").write_text(json.dumps(completion, indent=2))
with zipfile.ZipFile(BASE / "evaluation_outputs.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for p in sorted(OUT.iterdir()):
        if p.is_file():
            archive.write(p, p.name)
print("EVALUATION_COMPLETE", len(trials), "fits", flush=True)
